# Understand UAE from PyTorch's workflow

Yu Sun
25/02/2026

* The original code was built using MMCV, a framework developed by Aliababa. 
* The MMCV framework follows this philosophy:
    * Define things using config files
    * Makes it simple for comparison studies, e.g. everything else the same but change the model from nnUNet to ViT
    * Good for training, but bad for development. Essentially you're debugging config files.


The following code disect the MMCV components and represent it using the standard PyTorch workflow.
* This will be the first step for us -> understanding what it's doing and how it's connected. Then build on top of it.
* It follows the logics of PyTorch 
    * Model
    * Data
    * Training loop, i.e. how to get the loss

If you're not familiar with PyTorch's training loop, spend some time review it.

## How to run
* A light version of Docker image is available for development using CPU: `sunyu0410/uae_light`
* Set up with the following code. It downloads the weights and sample data.

    ```bash
    # Download weights and data
    git clone https://github.com/alibaba-damo-academy/self-supervised-anatomical-embedding-v2.git prj
    cd prj
    pip install gdown && \
        gdown 1LH9E5D273kOJXrUmBv_s2hXuOZV-dR65 -O weights.zip && \
        unzip weights.zip && \
        mv Self-supervised_Anatomical_Embeddings/checkpoints . && \
        mv Self-supervised_Anatomical_Embeddings/data . && \
        rm -r Self-supervised_Anatomical_Embeddings weights.zip 

    ```

* Install Ipython for better UI:
`pip install ipython`

* Preprocess the sample data:
`ipython misc/lymphnode_preprocess_crop_multi_process.py`

Now should be ready to continue.


## Model

The model is constructed by
* Read the config file
* Build the model using `build_model()`

In [1]:
from sam.apis.train import train_detector
from mmdet.datasets import build_dataset
from mmdet.models import build_detector
from mmcv import Config
from sam import *


/usr/local/lib/python3.7/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
cfg = Config.fromfile('configs/sam/sam_NIHLN.py')

The config will be read as a Python dictionary and contains all the arguments for the model. Once built, it's still a PyTorch model, as MMCV sits on top of PyTorch.

In [3]:
cfg.model

{'type': 'Sam',
 'backbone': {'type': 'ResNet3d',
  'pretrained2d': True,
  'pretrained': 'torchvision://resnet18',
  'depth': 18,
  'in_channels': 1,
  'spatial_strides': (2, 2, 2, 2),
  'temporal_strides': (1, 1, 1, 2),
  'conv1_kernel': (3, 7, 7),
  'conv1_stride_t': 1,
  'conv1_stride_s': 1,
  'pool1_stride_t': 1,
  'with_pool1': False,
  'with_pool2': True,
  'conv_cfg': {'type': 'Conv3d'},
  'inflate': ((0, 0), (0, 0), (1, 1), (1, 1)),
  'norm_eval': False,
  'zero_init_residual': False},
 'neck': {'type': 'FPN3d',
  'end_level': 3,
  'in_channels': [64, 128, 256],
  'out_channels': 128,
  'num_outs': 3,
  'conv_cfg': {'type': 'Conv3d'}},
 'read_out_head': {'type': 'FPN3d',
  'end_level': 1,
  'in_channels': [512],
  'out_channels': 128,
  'num_outs': 1,
  'conv_cfg': {'type': 'Conv3d'}},
 'train_cfg': {'pre_select_pos_number': 2000,
  'after_select_pos_number': 100,
  'pre_select_neg_number': 2000,
  'after_select_neg_number': 500,
  'positive_distance': 2.0,
  'ignore_distance'

In [4]:
model = build_detector(cfg.model)

2026-02-25 01:24:38,609 - mmdet - INFO - load model from: torchvision://resnet18
2026-02-25 01:24:38,648 - mmdet - WARNING - Module not exist in the state_dict_r2d: layer1.0.downsample.0
2026-02-25 01:24:38,649 - mmdet - WARNING - Module not exist in the state_dict_r2d: layer1.0.downsample.1
2026-02-25 01:24:38,781 - mmdet - INFO - These parameters in the 2d checkpoint are not loaded: {'fc.bias', 'fc.weight'}


load checkpoint from torchvision path: torchvision://resnet18


In [5]:
model

Sam(
  (backbone): ResNet3d(
    (conv1): ConvModule(
      (conv): Conv3d(1, 64, kernel_size=(3, 7, 7), stride=(1, 1, 1), padding=(1, 3, 3), bias=False)
      (bn): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (activate): ReLU(inplace=True)
    )
    (maxpool): MaxPool3d(kernel_size=(1, 3, 3), stride=(1, 2, 2), padding=(0, 1, 1), dilation=1, ceil_mode=False)
    (pool2): MaxPool3d(kernel_size=(2, 1, 1), stride=(2, 1, 1), padding=0, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock3d(
        (conv1): ConvModule(
          (conv): Conv3d(64, 64, kernel_size=(1, 3, 3), stride=(1, 2, 2), padding=(0, 1, 1), bias=False)
          (bn): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (activate): ReLU(inplace=True)
        )
        (conv2): ConvModule(
          (conv): Conv3d(64, 64, kernel_size=(1, 3, 3), stride=(1, 1, 1), padding=(0, 1, 1), bias=False)
          (bn): BatchNorm3d

## Data

* Similarly, the dataset is built using `build_dataset()`
* Here I manually unbox the data and simulate one batch, to pass into the model. 
* Note that it not only needs the image, but also the meta information. The meta contains information about the file, the cropping information, etc.

In [6]:
ds = build_dataset(cfg.data.train)


import torch
from mmcv.parallel import DataContainer

# 1. Get two samples from the dataset
data1 = ds[0][0]
data2 = ds[0][1]

def unbox(dc):
    """Extracts the tensor or list from an MMCV DataContainer."""
    return dc.data if isinstance(dc, DataContainer) else dc

# 2. Manually Batch (Stack) the tensors
# We combine sample 1 and sample 2 into a batch of size 2
batch_img = torch.stack([unbox(data1['img']), unbox(data2['img'])])
batch_meshgrid = torch.stack([unbox(data1['meshgrid']), unbox(data2['meshgrid'])])
batch_valid = torch.stack([unbox(data1['valid']), unbox(data2['valid'])])


# 3. Handle img_metas (It must be a list of dicts)
# unbox(data1['img_metas']) usually returns a single dict
batch_metas = [unbox(data1['img_metas']), unbox(data2['img_metas'])]

## Get the loss

* We then run the model and the get the loss.
* Once we have the loss, we can then use the PyTorch training loop on the parameter update.

In [9]:
model = model.train().float()
losses = model.forward_train(
    img=batch_img,
    img_metas=batch_metas,
    meshgrid=batch_meshgrid,
    valid=batch_valid
)



In [10]:
print(losses)

{'loss': tensor(9.9093, grad_fn=<AddBackward0>)}


# Task
Think about and explore
* What preprocessing steps are done on the image data (e.g. from NIfTI files to the unboxed batch)
* How is the loss calculated internally? Check the `sam/models/frameworks/sam.py` and the functions to calcualte losses.